In [1]:
import os
os.chdir('../../../../..')

In [ ]:
import polars as pl
import numpy as np
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

In [3]:
path = 'results/qm9/regression_benchmark/n_2000/raw_all_targets.csv'
df = pl.read_csv(path)

In [4]:
df

method,kind,best_alpha,best_beta,cv_rmse,total_seconds,train_rmse,test_rmse,test_mae,test_r2,status,seed,block,target
str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,i64,str,str
"""baseline_mean""","""control""",null,null,null,0.000071,null,1.388368,1.107919,-0.000556,"""ok""",42,"""control""","""gap"""
"""baseline_composition""","""control""",0.1,0.01,null,2.190921,null,1.19367,0.852676,0.260393,"""ok""",42,"""control""","""gap"""
"""morgan_tanimoto_direct""","""kernel""",0.1,null,0.909643,0.794806,0.117474,1.108333,0.62316,0.362364,"""ok""",42,"""reference""","""gap"""
"""morgan_tanimoto_laplacian""","""vector""",0.1,null,0.640326,0.849763,0.099901,0.822856,0.490986,0.648537,"""ok""",42,"""reference""","""gap"""
"""morgan_euclidean_laplacian""","""vector""",0.1,null,0.632987,2.214159,0.099735,0.768775,0.491277,0.693217,"""ok""",42,"""reference""","""gap"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mace_riemann_affine_invariant""","""distance""",0.1,null,0.9025,0.834959,0.167355,0.773092,0.58095,0.65714,"""ok""",456,"""controlled""","""mu"""
"""mace_ph_coordinates_bottleneck""","""distance""",5.0,null,1.314584,2.796042,1.229033,1.213982,0.95307,0.154567,"""ok""",456,"""controlled""","""mu"""
"""mace_ph_coordinates_sliced_was…","""distance""",0.1,null,1.143065,0.830892,0.522245,1.028187,0.785676,0.393545,"""ok""",456,"""controlled""","""mu"""


In [13]:
TABLE_13_SOAP = [
    ("soap_grassmann_chordal_dist_laplacian", "soap_grassmann_geodesic_dist_laplacian"),
    ("soap_grassmann_projection_laplacian", "soap_grassmann_geodesic_dist_laplacian")
]

TABLE_13_MACE = [
    ("mace_grassmann_chordal_dist_laplacian", "mace_grassmann_geodesic_dist_laplacian"),
    ("mace_grassmann_projection_laplacian", "mace_grassmann_geodesic_dist_laplacian")
]

# --- Table 10: The SPD Ladder (Rung-by-Rung Evolution) ---
TABLE_10_SOAP = []
TABLE_10_MACE = []

for kernel in ["linear", "rbf"]:
    # SOAP Ladder Rungs
    TABLE_10_SOAP.extend([
        ("soap_riemann_tangent_" + kernel, "soap_covariance_flat_" + kernel),
        ("soap_covariance_flat_" + kernel, "soap_spd_diagonal_" + kernel),
        ("soap_spd_diagonal_" + kernel, "soap_spd_scalar_" + kernel),
        ("soap_riemann_tangent_" + kernel, "soap_averaged_" + kernel),
    ])
    # MACE Ladder Rungs
    TABLE_10_MACE.extend([
        ("mace_riemann_tangent_" + kernel, "mace_covariance_flat_" + kernel),
        ("mace_covariance_flat_" + kernel, "mace_spd_diagonal_" + kernel),
        ("mace_spd_diagonal_" + kernel, "mace_spd_scalar_" + kernel),
        ("mace_riemann_tangent_" + kernel, "mace_averaged_" + kernel),
    ])


def run_isolated_tests(df: pl.DataFrame, pairs: list, table_label: str, desc_label: str):
    targets = df["target"].unique().to_list()

    for target in targets:
        df_target = df.filter(pl.col("target") == target)

        # Pivot to secure paired seed rows
        try:
            pivoted = df_target.pivot(
                on="method", index="seed", values="test_r2", aggregate_function=None
            )
        except Exception:
            continue

        raw_p_values = []
        valid_results = []

        for m1, m2 in pairs:
            if m1 not in pivoted.columns or m2 not in pivoted.columns:
                continue

            v1 = pivoted[m1].to_numpy()
            v2 = pivoted[m2].to_numpy()

            mask = ~np.isnan(v1) & ~np.isnan(v2)
            if np.sum(mask) < 2:
                continue

            # Paired parametric t-test
            _, p_val = ttest_rel(v1[mask], v2[mask])

            raw_p_values.append(p_val)
            valid_results.append({
                "Method A": m1.replace(f"{desc_label.lower()}_", ""),
                "Method B": m2.replace(f"{desc_label.lower()}_", ""),
                "Mean Δ R²": np.mean(v1[mask]) - np.mean(v2[mask])
            })

        if not raw_p_values:
            continue

        # Multiple testing correction isolated strictly to this sub-family/target block
        reject, corrected_p, _, _ = multipletests(raw_p_values, alpha=0.05, method="holm")

        out = []
        for res, p_r, p_c, sig in zip(valid_results, raw_p_values, corrected_p, reject):
            res["p-value (Holm)"] = p_c
            res["Significant?"] = "YES" if sig else "no"
            out.append(res)

        print(f"\n[+] {table_label} | {desc_label} | Target: {target.upper()}")
        print(pl.DataFrame(out).select(["Method A", "Method B", "Mean Δ R²", "p-value (Holm)", "Significant?"]))

In [14]:

print("\n##### PROCESSING TABLE 13 HYPOTHESES #####")
run_isolated_tests(df, TABLE_13_SOAP, "TABLE 13", "SOAP")
run_isolated_tests(df, TABLE_13_MACE, "TABLE 13", "MACE")

# --- TABLE 10 ---
print("\n##### PROCESSING TABLE 10 HYPOTHESES #####")
run_isolated_tests(df, TABLE_10_SOAP, "TABLE 10", "SOAP")
run_isolated_tests(df, TABLE_10_MACE, "TABLE 10", "MACE")


##### PROCESSING TABLE 13 HYPOTHESES #####

[+] TABLE 13 | SOAP | Target: MU
shape: (2, 5)
┌───────────────────────────┬──────────────────────────┬───────────┬────────────────┬──────────────┐
│ Method A                  ┆ Method B                 ┆ Mean Δ R² ┆ p-value (Holm) ┆ Significant? │
│ ---                       ┆ ---                      ┆ ---       ┆ ---            ┆ ---          │
│ str                       ┆ str                      ┆ f64       ┆ f64            ┆ str          │
╞═══════════════════════════╪══════════════════════════╪═══════════╪════════════════╪══════════════╡
│ grassmann_chordal_dist_la ┆ grassmann_geodesic_dist_ ┆ 0.019577  ┆ 0.095029       ┆ no           │
│ placi…                    ┆ laplac…                  ┆           ┆                ┆              │
│ grassmann_projection_lapl ┆ grassmann_geodesic_dist_ ┆ 0.019577  ┆ 0.095029       ┆ no           │
│ acian                     ┆ laplac…                  ┆           ┆                ┆              │

In [12]:
print(df.filter(pl.col("method").str.contains("grassmann"))["method"].unique().to_list())

['mace_grassmann_projection_laplacian', 'mace_grassmann_chordal_dist_laplacian', 'soap_grassmann_geodesic', 'soap_grassmann_projection_rbf', 'soap_grassmann_projection_linear', 'mace_grassmann_projection_linear', 'soap_grassmann_projection_laplacian', 'soap_grassmann_chordal_dist_laplacian', 'soap_grassmann_geodesic_dist_laplacian', 'mace_grassmann_geodesic_dist_laplacian', 'mace_grassmann_geodesic', 'mace_grassmann_projection_rbf']
